# 🚴 Food Delivery Delay Prediction - Exploratory Data Analysis (EDA)
**Author:** Aadi Shah | B.Tech CSE (23000255)  
**Algorithm:** Random Forest Regressor (`sklearn.ensemble.RandomForestRegressor`)  
**Task:** Exploratory analysis of 200,000 food delivery orders to understand delay dynamics, route efficiency, environmental factors, and feature correlations.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_PATH, TARGET_COLUMN
from src.data_loader import load_raw_data, validate_and_clean_data

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Environment and libraries configured successfully.')

## 1. Data Ingestion & Quality Validation
Load the raw 200,000 order records and perform data cleaning (Slide 4 rules):
- Zero missing values constraint
- Zero duplicate rows
- Removal of invalid/negative delays
- Removal of non-predictive identifiers (`restaurant_id`)

In [ ]:
raw_df = load_raw_data(filepath=RAW_DATA_PATH)
clean_df, summary = validate_and_clean_data(raw_df)

print(f"Initial records: {summary['initial_records']:,}")
print(f"Cleaned records: {summary['final_records']:,}")
print(f"Missing values: {summary['missing_values']}")
print(f"Duplicate rows: {summary['duplicate_rows']}")
clean_df.head()

## 2. Statistical Summary of Target & Numerical Predictors
Target variable `delivery_delay` is continuous (measured in minutes).

In [ ]:
clean_df[['delivery_delay', 'delivery_distance', 'order_value', 'route_efficiency', 'age', 'order_history']].describe()

## 3. Target Distribution (`delivery_delay`)
Examine the distribution shape, skewness, and range of delivery delays.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(clean_df['delivery_delay'], kde=True, bins=50, color='#2b5c8f', ax=ax)
ax.set_title('Distribution of Delivery Delays (Minutes)', fontsize=14, fontweight='bold')
ax.set_xlabel('Delivery Delay (minutes)')
ax.set_ylabel('Frequency')
plt.show()

## 4. Environmental Interactions: Traffic vs. Weather
Analyze compound delay impacts when adverse weather coincides with high traffic conditions.

In [ ]:
traffic_weather_pivot = clean_df.pivot_table(
    values='delivery_delay',
    index='traffic_condition',
    columns='weather_condition',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(traffic_weather_pivot, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': 'Mean Delay (min)'}, ax=ax)
ax.set_title('Mean Delivery Delay by Traffic and Weather Conditions', fontsize=13, fontweight='bold')
plt.show()

## 5. Route Efficiency & Distance Analysis
How does route efficiency mitigate delivery delays across various travel distances?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sample_viz = clean_df.sample(2000, random_state=42)
sns.scatterplot(
    data=sample_viz,
    x='delivery_distance',
    y='delivery_delay',
    hue='delivery_method',
    alpha=0.6,
    ax=ax
)
ax.set_title('Delivery Delay vs Distance by Delivery Vehicle Method', fontsize=13, fontweight='bold')
plt.show()

## 6. Correlation Matrix
Assess linear dependencies among continuous numerical variables.

In [ ]:
num_cols = ['delivery_delay', 'delivery_distance', 'order_value', 'route_efficiency', 'age', 'order_history', 'customer_rating']
corr = clean_df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Matrix of Numerical Features', fontsize=13, fontweight='bold')
plt.show()

## 7. Key Findings & Modeling Takeaways
1. **Route Efficiency & Distance**: Highest non-linear predictors of delay.
2. **Compound Environmental Bottlenecks**: Heavy rain / storms combined with high traffic compound delay times significantly.
3. **Vehicle Agility**: Bikes demonstrate more resilient travel times in congested urban zones compared to Cars.
4. **Non-linear Modeling Requirement**: Tree-based ensembles (Random Forest Regressor) are ideally suited to capture multi-way non-linear feature interactions without requiring artificial polynomial expansions.